<a href="https://colab.research.google.com/github/nekokaito/deep-learning-rooftop-plant-health/blob/main/rooftop_plant_health_classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import files

uploaded = files.upload()

Saving rooftop_plant_health.zip to rooftop_plant_health.zip


In [2]:
import zipfile
import os

zip_file = "rooftop_plant_health.zip"

with zipfile.ZipFile(zip_file, 'r') as zip_ref:
    zip_ref.extractall("/content/dataset")

print("Dataset extracted successfully!")

Dataset extracted successfully!


In [3]:
import os

dataset_path = "/content/dataset"

for root, dirs, files in os.walk(dataset_path):
    level = root.replace(dataset_path, '').count(os.sep)
    indent = ' ' * 2 * level
    print(indent + os.path.basename(root) + "/")

dataset/
  Dry_Severe_Damage/
  Healthy/
  Yellowing_Discoloration/
  Insect_Physical_Damage/
  Leaf_Spot_Browning/


In [4]:
import os

classes = [
    "Healthy",
    "Yellowing_Discoloration",
    "Leaf_Spot_Browning",
    "Dry_Severe_Damage",
    "Insect_Physical_Damage"
]

for class_name in classes:
    folder = os.path.join(dataset_path, class_name)

    if os.path.exists(folder):
        images = [
            f for f in os.listdir(folder)
            if f.lower().endswith((".jpg", ".jpeg", ".png"))
        ]

        print(f"{class_name}: {len(images)} images")
    else:
        print(f"{class_name}: folder not found")

Healthy: 31 images
Yellowing_Discoloration: 10 images
Leaf_Spot_Browning: 13 images
Dry_Severe_Damage: 6 images
Insect_Physical_Damage: 4 images


In [9]:
import os
import shutil
import random
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd


DATASET_DIR = Path("/content/dataset")


image_extensions = {".jpg", ".jpeg", ".png", ".webp"}


classes = [
    "Healthy",
    "Yellowing_Discoloration",
    "Leaf_Spot_Browning",
    "Dry_Severe_Damage",
    "Insect_Physical_Damage"
]

print("Setup completed!")

Setup completed!


In [10]:
OUTPUT_DIR = Path("/content/plant_dataset_split")


if OUTPUT_DIR.exists():
    shutil.rmtree(OUTPUT_DIR)

for split in ["train", "validation", "test"]:
    for class_name in classes:
        (OUTPUT_DIR / split / class_name).mkdir(
            parents=True,
            exist_ok=True
        )

print("Folders created successfully!")

Folders created successfully!


In [12]:
random.seed(42)

for class_name in classes:

    class_dir = DATASET_DIR / class_name

    images = [
        f for f in class_dir.iterdir()
        if f.is_file() and f.suffix.lower() in image_extensions
    ]


    random.shuffle(images)

    n = len(images)


    n_train = max(1, int(n * 0.70))
    n_val = max(1, int(n * 0.15))


    n_test = n - n_train - n_val

    if n_test < 1:
        n_test = 1
        n_train -= 1

    train_images = images[:n_train]
    val_images = images[n_train:n_train + n_val]
    test_images = images[n_train + n_val:]

    for image in train_images:
        shutil.copy2(
            image,
            OUTPUT_DIR / "train" / class_name / image.name
        )

    for image in val_images:
        shutil.copy2(
            image,
            OUTPUT_DIR / "validation" / class_name / image.name
        )

    for image in test_images:
        shutil.copy2(
            image,
            OUTPUT_DIR / "test" / class_name / image.name
        )

print("Dataset split completed!")

Dataset split completed!


In [13]:
for split in ["train", "validation", "test"]:

    print(f"\n===== {split.upper()} =====")

    total = 0

    for class_name in classes:

        folder = OUTPUT_DIR / split / class_name

        count = len([
            f for f in folder.iterdir()
            if f.is_file() and f.suffix.lower() in image_extensions
        ])

        total += count

        print(f"{class_name}: {count}")

    print("Total:", total)


===== TRAIN =====
Healthy: 21
Yellowing_Discoloration: 7
Leaf_Spot_Browning: 9
Dry_Severe_Damage: 4
Insect_Physical_Damage: 2
Total: 43

===== VALIDATION =====
Healthy: 4
Yellowing_Discoloration: 1
Leaf_Spot_Browning: 1
Dry_Severe_Damage: 1
Insect_Physical_Damage: 1
Total: 8

===== TEST =====
Healthy: 6
Yellowing_Discoloration: 2
Leaf_Spot_Browning: 3
Dry_Severe_Damage: 1
Insect_Physical_Damage: 1
Total: 13


In [14]:
records = []

for split in ["train", "validation", "test"]:

    for class_name in classes:

        folder = OUTPUT_DIR / split / class_name

        for image_file in folder.iterdir():

            if image_file.suffix.lower() in image_extensions:

                records.append({
                    "filename": image_file.name,
                    "class": class_name,
                    "split": split,
                    "path": str(image_file)
                })

split_df = pd.DataFrame(records)

print(split_df.head())
print("\nTotal records:", len(split_df))

                     filename    class  split  \
0  IMG_20260927_121257458.jpg  Healthy  train   
1  IMG_20260910_152732087.jpg  Healthy  train   
2  IMG_20260910_153058212.jpg  Healthy  train   
3  IMG_20260927_121446567.jpg  Healthy  train   
4  IMG_20260927_121138086.jpg  Healthy  train   

                                                path  
0  /content/plant_dataset_split/train/Healthy/IMG...  
1  /content/plant_dataset_split/train/Healthy/IMG...  
2  /content/plant_dataset_split/train/Healthy/IMG...  
3  /content/plant_dataset_split/train/Healthy/IMG...  
4  /content/plant_dataset_split/train/Healthy/IMG...  

Total records: 64
